In [ ]:
import os
import cv2
import warnings
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Video
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
#h--- Set to True to compute simple features; keep False if features are already precomputed.
extract_features = False  

# False → use the compressed dataset (~130 GB), faster and lighter on resources.
use_full = False  

In [6]:
import os

VIDEO_ROOT = "/Data/barbados/barbados-traffic-analysis-challenge/data/videos"

# Heuristics to map a flat list of files back to logical datasets
LABEL_PATTERNS = {
    "n1": ["normanniles1", "norman_niles1", "normanNiles1", "n1_"],
    "n2": ["normanniles2", "norman_niles2", "normanNiles2", "n2_"],
    "n3": ["normanniles3", "norman_niles3", "normanNiles3", "n3_"],
    "n4": ["normanniles4", "norman_niles4", "normanNiles4", "n4_"],
    "added": ["added"],  # adjust if your "added" files have a specific pattern
}

def classify_label(filename: str) -> str:
    """Return logical label (n1, n2, n3, n4, added) based on filename."""
    lower = filename.lower()
    for label, patterns in LABEL_PATTERNS.items():
        if any(p in lower for p in patterns):
            return label
    return "unknown"


def labeled_folder_stats(root):
    stats = {label: {"files": 0, "bytes": 0} for label in list(LABEL_PATTERNS.keys()) + ["unknown"]}

    for r, _, files in os.walk(root):
        for f in files:
            if not f.lower().endswith(".mp4"):
                continue
            fp = os.path.join(r, f)
            label = classify_label(f)
            stats[label]["files"] += 1
            stats[label]["bytes"] += os.path.getsize(fp)

    # Print in the same style
    print("--- COMPRESSED DATASETS ---")
    total_files = 0
    total_gb = 0.0
    for label in ["n1", "n2", "n3", "n4", "added"]:
        files = stats[label]["files"]
        gb = stats[label]["bytes"] / (1024 ** 3)
        total_files += files
        total_gb += gb
        print(f"{label}: {files} files, {gb:.2f} GB")
    print(f"TOTAL: {total_files} files, {total_gb:.2f} GB")

    # Optional: show any unmatched files
    unk_files = stats["unknown"]["files"]
    if unk_files > 0:
        unk_gb = stats["unknown"]["bytes"] / (1024 ** 3)
        print(f"\nWARNING: {unk_files} files ({unk_gb:.2f} GB) classified as 'unknown' – "
              f"adjust LABEL_PATTERNS if needed.")


# Run
labeled_folder_stats(VIDEO_ROOT)

--- COMPRESSED DATASETS ---
n1: 3253 files, 23.70 GB
n2: 3253 files, 22.16 GB
n3: 3253 files, 23.64 GB
n4: 3253 files, 22.98 GB
added: 0 files, 0.00 GB
TOTAL: 13012 files, 92.48 GB


In [13]:
import os
import cv2
import warnings

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# ============================================================
# Local config (NO Kaggle)
# ============================================================

# Folder where your CSVs live
DATA_ROOT = "/Data/barbados/barbados-traffic-analysis-challenge/data"

# Folder where ALL compressed videos live (possibly with subfolders)
VIDEO_ROOT = "/Data/barbados/barbados-traffic-analysis-challenge/data/videos"

# Toggle this to extract video features (do it once, then reuse CSV)
extract_features = True          # set False after first successful run
features_csv_path = "video_features.csv"

# sampling frames from each ~60s clip
SAMPLE_INTERVAL = 60             # 1 frame every 60 frames (~0.5 FPS if 30 FPS)

RANDOM_STATE = 42
N_ESTIMATORS = 400
MAX_DEPTH = None
N_JOBS = -1

# ============================================================
# Resolve CSV paths (Train / Test / SampleSubmission)
# ============================================================

# Try Train-2.csv first, fall back to Train.csv
train_candidates = ["Train-2.csv", "Train.csv"]
TRAIN_CSV = None
for fname in train_candidates:
    path = os.path.join(DATA_ROOT, fname)
    if os.path.exists(path):
        TRAIN_CSV = path
        break

if TRAIN_CSV is None:
    raise FileNotFoundError(
        f"Could not find Train-2.csv or Train.csv under {DATA_ROOT}"
    )

TEST_CSV = os.path.join(DATA_ROOT, "TestInputSegments.csv")
SUB_CSV  = os.path.join(DATA_ROOT, "SampleSubmission.csv")

for p in [TEST_CSV, SUB_CSV]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing required CSV: {p}")


# ============================================================
# Build video index from your single VIDEO_ROOT
# ============================================================

def build_video_index_from_root(root):
    """
    Scan VIDEO_ROOT and build a mapping:
        videos (file id without extension) -> full path
    """
    records = []
    for r, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(".mp4"):
                file_id = os.path.splitext(f)[0]
                file_path = os.path.join(r, f)
                records.append({"videos": file_id, "video_path": file_path})
    if not records:
        raise FileNotFoundError(
            f"No MP4 files found under {root}. "
            f"Check VIDEO_ROOT or that the files are actually there."
        )
    df = pd.DataFrame(records)
    # Optional sanity check:
    print(f"VIDEO_INDEX: {df['videos'].nunique()} unique video ids, {len(df)} files")
    return df

VIDEO_INDEX = build_video_index_from_root(VIDEO_ROOT)


# ============================================================
# Load CSVs and join with local video paths
# ============================================================

train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SUB_CSV)

# normalise 'videos' to bare file id (strip folder + .mp4)
for df in (train, test):
    df["videos"] = df["videos"].apply(
        lambda x: os.path.splitext(os.path.basename(str(x)))[0]
    )

# attach local video paths
train = train.merge(VIDEO_INDEX, on="videos", how="left")
test  = test.merge(VIDEO_INDEX, on="videos", how="left")

# quick diagnostics
if train["video_path"].isnull().any() or test["video_path"].isnull().any():
    print("WARNING: some rows could not be matched to a local video file.")
    missing_train = train[train["video_path"].isnull()]["videos"].unique()
    missing_test  = test[test["video_path"].isnull()]["videos"].unique()
    print("Missing in train (first 10):", missing_train)
    print("Missing in test  (first 10):", missing_test)
else:
    print("All train/test rows successfully matched to local video files.")

VIDEO_INDEX: 13012 unique video ids, 13012 files
Missing in train (first 10): []
Missing in test  (first 10): ['normanniles1_2025-10-20-09-01-45' 'normanniles1_2025-10-20-09-02-45'
 'normanniles1_2025-10-20-09-03-45' ... 'normanniles4_2025-10-26-13-32-45'
 'normanniles4_2025-10-26-13-33-45' 'normanniles4_2025-10-26-13-34-45']
